# Installing Dependencies


과제를 수행함에 있어 설치해야 하는 의존성 라이브러리 등이 있을 시 아래 코드 셀에 설치 명령을 작성할 것

In [2]:
%pip install lightgbm scikit-learn pandas numpy scipy

  Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl.metadata (17 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl (1.6 MB)
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl (8.1 MB)
Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)
Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl (20.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━

# Data Load


훈련 입력 데이터: **TRAIN_DATA**, 훈련 레이블: **TRAIN_LABEL**, 테스트 입력 데이터: **TEST_DATA**

In [3]:
import pandas as pd

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)

TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)


## 1. Import Libraries & Load Test Label

In [4]:
import numpy as np
import lightgbm as lgb
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load test label (contains id and timestamp for prediction)
TEST_LABEL = pd.read_csv('test-label.csv')

# Fix potential trailing spaces in test timestamps
TEST_LABEL['timestamp'] = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)
TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)

print('TRAIN_LABEL stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())
print()
print('TEST_LABEL shape:', TEST_LABEL.shape)

TRAIN_LABEL stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64

TEST_LABEL shape: (1028, 4)


## 2. Feature Extraction from Time-Series Sensor Data

각 레이블의 timestamp를 기준으로 이전 **3분(180,000ms)** 구간의 센서 데이터를 집계하여 통계적 특성값을 추출합니다.

추출하는 통계량: mean, std, min, max, median, skewness, kurtosis, range, Q25, Q75, IQR

추가 특성: 가속도계 3축 합성 크기(magnitude)

In [5]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']
WINDOW_MS   = 180_000  # 3분 윈도우 (레이블 간격과 동일)

def extract_features(label_df, sensor_df):
    """label_df의 각 (pid, timestamp)에 대해 sensor_df에서 특성값을 추출."""
    sensor_by_pid = {
        pid: grp.sort_values('timestamp').reset_index(drop=True)
        for pid, grp in sensor_df.groupby('pid')
    }

    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']
        ts  = float(lrow['timestamp'])
        lid = lrow['id']

        feat = {'id': lid}

        if pid not in sensor_by_pid:
            # pid가 없으면 모두 NaN
            for c in SENSOR_COLS:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr']:
                    feat[f'{c}_{s}'] = np.nan
            feat['accel_mag_mean'] = np.nan
            feat['accel_mag_std']  = np.nan
            feat['accel_mag_max']  = np.nan
            rows.append(feat)
            continue

        sg   = sensor_by_pid[pid]
        mask = (sg['timestamp'] >= ts - WINDOW_MS) & (sg['timestamp'] <= ts)
        win  = sg.loc[mask, SENSOR_COLS]

        if len(win) == 0:
            for c in SENSOR_COLS:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr']:
                    feat[f'{c}_{s}'] = np.nan
            feat['accel_mag_mean'] = np.nan
            feat['accel_mag_std']  = np.nan
            feat['accel_mag_max']  = np.nan
        else:
            for c in SENSOR_COLS:
                v = win[c].values.astype(float)
                feat[f'{c}_mean']   = np.mean(v)
                feat[f'{c}_std']    = np.std(v)
                feat[f'{c}_min']    = np.min(v)
                feat[f'{c}_max']    = np.max(v)
                feat[f'{c}_median'] = np.median(v)
                feat[f'{c}_skew']   = stats.skew(v)        if len(v) > 2 else 0.0
                feat[f'{c}_kurt']   = stats.kurtosis(v)    if len(v) > 2 else 0.0
                feat[f'{c}_range']  = np.max(v) - np.min(v)
                feat[f'{c}_q25']    = np.percentile(v, 25)
                feat[f'{c}_q75']    = np.percentile(v, 75)
                feat[f'{c}_iqr']    = np.percentile(v, 75) - np.percentile(v, 25)

            # 3축 가속도 합성 크기
            mag = np.sqrt(win['accel_x'].values**2 +
                          win['accel_y'].values**2 +
                          win['accel_z'].values**2)
            feat['accel_mag_mean'] = np.mean(mag)
            feat['accel_mag_std']  = np.std(mag)
            feat['accel_mag_max']  = np.max(mag)

        rows.append(feat)

    return pd.DataFrame(rows).set_index('id')

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA)
print(f'  → shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA)
print(f'  → shape: {test_features.shape}')

Extracting train features...
  → shape: (815, 69)
Extracting test features...
  → shape: (1028, 69)


In [6]:
print("Test NaN rows:", test_features.isna().all(axis=1).sum(), "out of", len(test_features))

Test NaN rows: 0 out of 1028


In [9]:
print("Train sensor ts range:", TRAIN_DATA['timestamp'].min(), "–", TRAIN_DATA['timestamp'].max())
print("Test sensor ts range:", TEST_DATA['timestamp'].min(), "–", TEST_DATA['timestamp'].max())
print("Test label ts range:", TEST_LABEL['timestamp'].min(), "–", TEST_LABEL['timestamp'].max())

Train sensor ts range: 1586903460000.0 – 1607736959968.75
Test sensor ts range: 1587215700000.0 – 1607846459968.75
Test label ts range: 1587215880000.0 – 1607846460000.0


In [10]:
print("Test PIDs missing from sensor:", set(TEST_LABEL['pid'].unique()) - set(TEST_DATA['pid'].unique()))

Test PIDs missing from sensor: set()


## 3. Prepare X, y and Handle Missing Values

In [7]:
# PID를 숫자로 인코딩하여 특성에 추가
train_label_indexed = TRAIN_LABEL.set_index('id')
test_label_indexed  = TEST_LABEL.set_index('id')

pid_map_tr = {p: i for i, p in enumerate(train_label_indexed['pid'].unique())}
pid_map_te = {p: i for i, p in enumerate(test_label_indexed['pid'].unique())}

X      = train_features.copy()
X_test = test_features.copy()

X['pid_enc']      = train_label_indexed['pid'].map(pid_map_tr).fillna(-1)
X_test['pid_enc'] = test_label_indexed['pid'].map(pid_map_te).fillna(-1)

y = train_label_indexed['stress'].astype(int)

# Median imputation for any NaN rows
imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(X),      columns=X.columns,      index=X.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test),     columns=X_test.columns, index=X_test.index)

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class distribution:', y.value_counts().sort_index().to_dict())

X_imp shape     : (815, 70)
X_test_imp shape: (1028, 70)
Class distribution: {0: 162, 1: 66, 2: 587}


## 4. Model Training — LightGBM with Class Weights + Multi-Seed Ensemble

- **LightGBM** (Gradient-Boosted Trees): 표 형태 데이터에서 최고 성능
- **class_weight='balanced'** + sample_weight: 불균형 레이블(0:162, 1:66, 2:587) 처리
- **Stratified 5-Fold CV**: 클래스 비율 유지하며 일반화 성능 추정
- **Multi-seed soft-voting ensemble**: 3개의 랜덤 시드로 학습 후 평균 확률값으로 최종 예측

In [8]:
# Class sample weights (Balanced Accuracy 최적화)
counts         = Counter(y)
total          = len(y)
n_cls          = len(counts)
class_weights  = {c: total / (n_cls * cnt) for c, cnt in counts.items()}
sample_weights = np.array([class_weights[yi] for yi in y])
print('Class weights:', class_weights)

# Best LightGBM hyperparameters (found via grid search)
LGBM_PARAMS = dict(
    n_estimators     = 1000,
    learning_rate    = 0.02,
    num_leaves       = 127,
    max_depth        = -1,
    min_child_samples= 5,
    subsample        = 0.6,
    colsample_bytree = 0.6,
    reg_alpha        = 0.3,
    reg_lambda       = 0.3,
    class_weight     = 'balanced',
    objective        = 'multiclass',
    num_class        = 3,
    n_jobs           = -1,
    verbose          = -1,
)

SEEDS          = [42, 7, 123]
all_test_proba = []
all_cv_scores  = []

for seed in SEEDS:
    skf            = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    test_proba_acc = np.zeros((len(X_test_imp), 3))
    fold_scores    = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr, X_val = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx],     y.iloc[val_idx]
        sw_tr        = sample_weights[tr_idx]

        model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        model.fit(
            X_tr, y_tr,
            sample_weight = sw_tr,
            eval_set      = [(X_val, y_val)],
            callbacks     = [lgb.early_stopping(100, verbose=False),
                             lgb.log_evaluation(-1)],
        )

        val_preds = model.predict(X_val)
        score     = balanced_accuracy_score(y_val, val_preds)
        fold_scores.append(score)
        test_proba_acc += model.predict_proba(X_test_imp)

    test_proba_acc /= 5
    all_test_proba.append(test_proba_acc)

    mean_s = np.mean(fold_scores)
    all_cv_scores.append(mean_s)
    print(f'Seed {seed}: CV Balanced Accuracy = {mean_s:.4f} ± {np.std(fold_scores):.4f}')

print(f'\nEnsemble CV Balanced Accuracy: {np.mean(all_cv_scores):.4f}')

Class weights: {1: 4.116161616161616, 0: 1.676954732510288, 2: 0.46280522430437254}


KeyboardInterrupt: 

## 5. Generate Final Predictions

In [ ]:
# Soft-voting: average probabilities across seeds, then argmax
final_proba = np.mean(all_test_proba, axis=0)
final_preds = np.argmax(final_proba, axis=1).astype(int)

print('Prediction distribution:')
unique, counts_pred = np.unique(final_preds, return_counts=True)
for u, c in zip(unique, counts_pred):
    print(f'  class {u}: {c}')

Prediction distribution:
  class 0: 302
  class 1: 75
  class 2: 651


# Generating Final Submissions


 Kaggle 제출물인 **csv 파일**을 생성하는 코드를 아래에 작성할 것

In [ ]:
submission = pd.DataFrame({
    'id':     TEST_LABEL['id'].values,
    'stress': final_preds,
})

submission.to_csv('submission_a1.csv', index=False)
print('submission.csv saved!')
print(submission.head(10))

submission.csv saved!
     id  stress
0  1227       2
1  1228       1
2  1229       2
3  1230       2
4  1231       1
5  1232       0
6  1233       0
7  1234       2
8  1235       0
9  1236       1
